In [1]:
# ==============================================================================
# NEOCORE COMPUTER LAB — SIMULADOR FUNCIONAL DE ARQUITETURA DE COMPUTADORES
# Disciplina: Computer Architecture and Organization | UNIFECAF
# Aluno: Ricardo Ruiz | Orientador: Prof. Vitor Vieira | Caxias do Sul - 2026
# ==============================================================================

class ControlBus:
    """
    Barramento de Controle (Control Bus):
    Sinais de controle emitidos pela Unidade de Controle (Hardwired).
    Sincroniza leitura/escrita na memória, registradores, E/S e seleção da ULA.
    """
    def __init__(self):
        self.MEM_READ = False
        self.MEM_WRITE = False
        self.IO_SELECT = False
        self.RegRead = False
        self.RegWrite = False
        self.FlagWrite = False
        self.ALUOp = "0000"
        self.HALT = False

    def reset_signals(self):
        self.MEM_READ = False
        self.MEM_WRITE = False
        self.IO_SELECT = False
        self.RegRead = False
        self.RegWrite = False
        self.FlagWrite = False
        self.ALUOp = "0000"

    def status_summary(self):
        active = []
        if self.MEM_READ: active.append("MEM_READ")
        if self.MEM_WRITE: active.append("MEM_WRITE")
        if self.IO_SELECT: active.append("IO_SELECT")
        if self.RegRead: active.append("RegRead")
        if self.RegWrite: active.append("RegWrite")
        if self.FlagWrite: active.append("FlagWrite")
        if self.HALT: active.append("HALT")
        active.append(f"ALUOp={self.ALUOp}")
        return " | ".join(active)


class DataBus:
    """
    Barramento de Dados (Data Bus):
    Bidirecional, 16 bits. Trafega dados e instruções entre CPU, RAM e Periféricos.
    """
    def __init__(self):
        self._value = 0x0000

    def write(self, data):
        self._value = data & 0xFFFF

    def read(self):
        return self._value


class AddressBus:
    """
    Barramento de Endereços (Address Bus):
    Unidirecional (CPU -> Memória), 12 bits (endereça até 4096 posições).
    """
    def __init__(self):
        self._address = 0x000

    def set_address(self, addr):
        self._address = addr & 0xFFF

    def get_address(self):
        return self._address


class DisplayIO:
    """Dispositivo de Saída Mapeado em Memória (Display Hexadecimal a 0xF01)."""
    def __init__(self):
        self.value = 0x0000

    def write(self, data):
        self.value = data & 0xFFFF
        print(f"\n   [PERIFÉRICO - DISPLAY (0xF01)] Valor Atualizado: 0x{self.value:04X} ({self.value} em decimal)")


class KeyboardIO:
    """Dispositivo de Entrada Mapeado em Memória (Teclado a 0xF00)."""
    def __init__(self):
        self.buffer = 0x0041 # Caractere 'A' em ASCII

    def read(self):
        return self.buffer


class CacheL1:
    """
    Memória Cache L1:
    - 64 linhas, Mapeamento Direto (Index = bits 5-0, Tag = bits 11-6).
    - Política de Escrita: Write-Through.
    """
    def __init__(self):
        self.lines = [{"valid": False, "tag": 0, "data": 0x0000} for _ in range(64)]
        self.hits = 0
        self.misses = 0

    def read(self, address):
        index = address & 0x3F         # Bits 5-0 (6 bits)
        tag = (address >> 6) & 0x3F    # Bits 11-6 (6 bits)
        line = self.lines[index]

        if line["valid"] and line["tag"] == tag:
            self.hits += 1
            return line["data"], True
        else:
            self.misses += 1
            return None, False

    def write(self, address, data):
        index = address & 0x3F
        tag = (address >> 6) & 0x3F
        self.lines[index] = {"valid": True, "tag": tag, "data": data & 0xFFFF}


class MemorySystem:
    """Subsistema de Memória (RAM 4K x 16 + Cache L1 + E/S Mapeada)."""
    def __init__(self, display, keyboard):
        self.ram = [0x0000] * 4096
        self.cache = CacheL1()
        self.display = display
        self.keyboard = keyboard

    def read(self, address, io_select=False):
        addr = address & 0xFFF

        # Faixa de E/S Mapeada em Memória (0xF00 - 0xF7F)
        if io_select or (0xF00 <= addr <= 0xF7F):
            if addr == 0xF00:
                print("   [E/S MAPEADA] Leitura do Teclado (0xF00)")
                return self.keyboard.read()
            elif addr == 0xF01:
                return self.display.value
            return 0x0000

        # Leitura com hierarquia de memória (Cache -> RAM)
        data, hit = self.cache.read(addr)
        if hit:
            print(f"   [CACHE L1 - HIT] Endereço 0x{addr:03X} encontrado na Cache -> Dado: 0x{data:04X}")
            return data
        else:
            print(f"   [CACHE L1 - MISS] Endereço 0x{addr:03X} não está na Cache. Buscando na RAM...")
            ram_data = self.ram[addr]
            self.cache.write(addr, ram_data)
            return ram_data

    def write(self, address, data, io_select=False):
        addr = address & 0xFFF

        if io_select or (0xF00 <= addr <= 0xF7F):
            if addr == 0xF01:
                self.display.write(data)
            return

        # Escrita em RAM e atualização na Cache (Write-Through)
        self.ram[addr] = data & 0xFFFF
        self.cache.write(addr, data)


class ULA:
    """Unidade Lógica e Aritmética (ULA / ALU) de 16 bits + Flags (Z, N, C, O)."""
    def __init__(self):
        self.flags = {"Z": 0, "N": 0, "C": 0, "O": 0}

    def execute(self, alu_op, operand_a, operand_b):
        result = 0
        if alu_op == "0100":  # ADD (A + B)
            raw = operand_a + operand_b
            result = raw & 0xFFFF
            
            self.flags["Z"] = 1 if result == 0 else 0
            self.flags["N"] = 1 if (result & 0x8000) != 0 else 0
            self.flags["C"] = 1 if raw > 0xFFFF else 0
            
            signed_a = operand_a if operand_a < 0x8000 else operand_a - 0x10000
            signed_b = operand_b if operand_b < 0x8000 else operand_b - 0x10000
            signed_res = signed_a + signed_b
            self.flags["O"] = 1 if signed_res < -32768 or signed_res > 32767 else 0

        elif alu_op == "0101":  # SUB (A - B)
            raw = operand_a - operand_b
            result = raw & 0xFFFF
            self.flags["Z"] = 1 if result == 0 else 0
            self.flags["N"] = 1 if (result & 0x8000) != 0 else 0
            self.flags["C"] = 1 if operand_a < operand_b else 0
            
            signed_a = operand_a if operand_a < 0x8000 else operand_a - 0x10000
            signed_b = operand_b if operand_b < 0x8000 else operand_b - 0x10000
            signed_res = signed_a - signed_b
            self.flags["O"] = 1 if signed_res < -32768 or signed_res > 32767 else 0

        elif alu_op == "0000":  # AND
            result = (operand_a & operand_b) & 0xFFFF
            self.flags["Z"] = 1 if result == 0 else 0
            self.flags["N"] = 1 if (result & 0x8000) != 0 else 0

        elif alu_op == "0001":  # OR
            result = (operand_a | operand_b) & 0xFFFF
            self.flags["Z"] = 1 if result == 0 else 0
            self.flags["N"] = 1 if (result & 0x8000) != 0 else 0

        return result


class CPU:
    """Unidade Central de Processamento (CPU NeoCore-16)."""
    def __init__(self, memory_system, data_bus, addr_bus, ctrl_bus):
        self.mem = memory_system
        self.data_bus = data_bus
        self.addr_bus = addr_bus
        self.ctrl_bus = ctrl_bus

        # Registradores Internos
        self.PC = 0x000     # Program Counter (12 bits)
        self.IR = 0x0000    # Instruction Register (16 bits)
        self.MAR = 0x000    # Memory Address Register (12 bits)
        self.MBR = 0x0000   # Memory Buffer Register (16 bits)

        # Banco de Registradores de Uso Geral (R0-R3)
        self.R = [0x0000, 0x0000, 0x0000, 0x0000]

        # ULA
        self.ula = ULA()

    def print_state(self, micro_step_title):
        print(f"\n{'-'*25} {micro_step_title} {'-'*25}")
        print(f" [REGISTRADORES INTERNOS] PC: 0x{self.PC:03X} | MAR: 0x{self.MAR:03X} | MBR: 0x{self.MBR:04X} | IR: 0x{self.IR:04X}")
        print(f" [REGS USO GERAL]        R0: 0x{self.R[0]:04X} | R1: 0x{self.R[1]:04X} | R2: 0x{self.R[2]:04X} | R3: 0x{self.R[3]:04X}")
        print(f" [FLAGS ULA]              Z={self.ula.flags['Z']} | N={self.ula.flags['N']} | C={self.ula.flags['C']} | O={self.ula.flags['O']}")
        print(f" [SINAIS DE CONTROLE]     {self.ctrl_bus.status_summary()}")

    def execute_clock_cycle(self):
        if self.ctrl_bus.HALT:
            print("\n[CPU] Processador em HALT. Execução interrompida.")
            return False

        # --- FASE 1: BUSCA (FETCH) ---
        # T0: MAR <- PC
        self.ctrl_bus.reset_signals()
        self.MAR = self.PC
        self.addr_bus.set_address(self.MAR)
        self.print_state("FETCH (T0): MAR <- PC")

        # T1: MBR <- RAM[MAR] e PC <- PC + 1
        self.ctrl_bus.MEM_READ = True
        data = self.mem.read(self.addr_bus.get_address(), self.ctrl_bus.IO_SELECT)
        self.data_bus.write(data)
        self.MBR = self.data_bus.read()
        self.PC = (self.PC + 1) & 0xFFF
        self.print_state("FETCH (T1): MBR <- RAM[MAR] | PC <- PC + 1")

        # T2: IR <- MBR
        self.IR = self.MBR
        self.print_state("FETCH (T2): IR <- MBR")

        # --- FASE 2: DECODIFICAÇÃO (DECODE) ---
        # T3: Decodificação do Opcode no IR
        opcode = (self.IR >> 12) & 0x0F
        rd = (self.IR >> 10) & 0x03
        rs = (self.IR >> 8) & 0x03
        imm_addr = self.IR & 0x0FFF

        if opcode == 0x1: # ADD Rd, Rs
            self.ctrl_bus.RegRead = True
            self.ctrl_bus.ALUOp = "0100"
            self.print_state(f"DECODE (T3): ADD R{rd}, R{rs} decodificado (Opcode=0x1)")
        elif opcode == 0x7: # STORE Rs, addr
            self.ctrl_bus.MEM_WRITE = True
            if (imm_addr & 0xFFF) >= 0xF00:
                self.ctrl_bus.IO_SELECT = True
            self.print_state(f"DECODE (T3): STORE R{rs}, [0x{imm_addr & 0xFFF:03X}] decodificado")
        elif opcode == 0xF: # HLT
            self.ctrl_bus.HALT = True
            self.print_state("DECODE (T3): HLT decodificado")
            return True

        # --- FASE 3: EXECUÇÃO (EXECUTE) ---
        if opcode == 0x1: # ADD Rd, Rs
            # T4: ULA opera A + B
            operand_a = self.R[rd]
            operand_b = self.R[rs]
            alu_out = self.ula.execute(self.ctrl_bus.ALUOp, operand_a, operand_b)
            self.print_state(f"EXECUTE (T4): ULA calcula 0x{operand_a:04X} + 0x{operand_b:04X} = 0x{alu_out:04X}")

            # T5: Gravação no registrador de destino
            self.ctrl_bus.RegWrite = True
            self.ctrl_bus.FlagWrite = True
            self.R[rd] = alu_out
            self.print_state(f"EXECUTE (T5): R{rd} <- 0x{alu_out:04X} | Flags atualizadas")

        elif opcode == 0x7: # STORE Rs, addr
            target_addr = imm_addr & 0xFFF
            self.MAR = target_addr
            self.addr_bus.set_address(self.MAR)
            self.MBR = self.R[rs]
            self.data_bus.write(self.MBR)
            self.mem.write(self.addr_bus.get_address(), self.data_bus.read(), self.ctrl_bus.IO_SELECT)
            self.print_state(f"EXECUTE (T4-T5): Memória/E/S[0x{target_addr:03X}] <- R{rs} (Valor: 0x{self.R[rs]:04X})")

        return True


# ==============================================================================
# DEMONSTRAÇÃO DO ESTUDO DE CASO PRÁTICO
# ==============================================================================
if __name__ == "__main__":
    display = DisplayIO()
    keyboard = KeyboardIO()
    memory_sys = MemorySystem(display, keyboard)
    data_bus = DataBus()
    addr_bus = AddressBus()
    ctrl_bus = ControlBus()
    cpu = CPU(memory_sys, data_bus, addr_bus, ctrl_bus)

    # Configuração Inicial
    cpu.PC = 0x00A      # Endereço inicial da instrução
    cpu.R[1] = 0x0005   # Registrador R1 = 5
    cpu.R[2] = 0x0003   # Registrador R2 = 3

    # Programa: ADD R1, R2 (0x1600), STORE R1 no Display (0x7401), HLT (0xF000)
    memory_sys.ram[0x00A] = 0x1600 
    memory_sys.ram[0x00B] = 0x7401 
    memory_sys.ram[0x00C] = 0xF000 

    # Execução
    cpu.execute_clock_cycle()


------------------------- FETCH (T0): MAR <- PC -------------------------
 [REGISTRADORES INTERNOS] PC: 0x00A | MAR: 0x00A | MBR: 0x0000 | IR: 0x0000
 [REGS USO GERAL]        R0: 0x0000 | R1: 0x0005 | R2: 0x0003 | R3: 0x0000
 [FLAGS ULA]              Z=0 | N=0 | C=0 | O=0
 [SINAIS DE CONTROLE]     ALUOp=0000
   [CACHE L1 - MISS] Endereço 0x00A não está na Cache. Buscando na RAM...

------------------------- FETCH (T1): MBR <- RAM[MAR] | PC <- PC + 1 -------------------------
 [REGISTRADORES INTERNOS] PC: 0x00B | MAR: 0x00A | MBR: 0x1600 | IR: 0x0000
 [REGS USO GERAL]        R0: 0x0000 | R1: 0x0005 | R2: 0x0003 | R3: 0x0000
 [FLAGS ULA]              Z=0 | N=0 | C=0 | O=0
 [SINAIS DE CONTROLE]     MEM_READ | ALUOp=0000

------------------------- FETCH (T2): IR <- MBR -------------------------
 [REGISTRADORES INTERNOS] PC: 0x00B | MAR: 0x00A | MBR: 0x1600 | IR: 0x1600
 [REGS USO GERAL]        R0: 0x0000 | R1: 0x0005 | R2: 0x0003 | R3: 0x0000
 [FLAGS ULA]              Z=0 | N=0 | C=0 | O=